In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://www.enjoytokyo.jp/event/list/sta200101/its04/',
   'title': '今日行ける！東京駅周辺のおすすめイベント',
   'content': '# 今日行ける！東京駅周辺のおすすめイベント. 2026/04/17(金) ～ 04/19(日) オープニングセレモニー：2026/04/18(土) 16:00～16:30. 昼はアート、夜は舞台。東京・丸の内が“花の祝祭空間”に。花絵25周年とポケモン30周年特別企画「花祝い」開催！. 2025/03/18(火) ～ 2027/03/31(水). 2026/04/03(金) ～ 09/30(水). 2026/04/13(月) ～ 04/24(金). 食材の宝庫“愛媛県”都市伝説の「蛇口からみかんジュース」が楽しめる公式キッチンカーも登場！大手町エリアへの賑わいを創出するイベントが開催。. 2026/03/21(土) ～ 05/18(月) ※各イベントによって開催日が異なります。詳細は公式サイトをご確認ください。. 2026/02/07(土) ～ 05/24(日) 休館日：2026/02/16(月)、03/16(月)、04/13(月)、05/11(月). モネ没後100年「クロード・モネ －風景への問いかけ」が、アーティゾン美術館にて開催されます。. 2026/04/06(月) ～ 04/26(日). XEX ATAGO GREEN HILLS ／ XEX TOKYO ／ XEX 日本橋：2026/1/15(木)～終了日未定. XEX WEST：2026/2/1(日)～終了日未定. 2026/02/01(日) ～ 05/31(日). リレー方式ラーメン企画「POPUPラーメン」第10弾はミシュランガイド愛知・岐阜・三重でビブグルマンに選出された「らぁめん登里勝（とりかつ）」。. 2026/04/01(水) ～ 05/31(日). 2026/04/01(水) ～ 05/31(日). 東京ビアホール＆ビアテラス14『TOKYO BEER WAVE』. 東京ビアホール＆ビアテラス14『TOKYO BEER WAVE』. ビールの波に、飲まれて飲んで！丸の内「東京ビアホール＆ビアテラス14」より、『TOKYO BEER W

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [9]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。東京都の面積は約2,194平方キロメートルですが、沖縄県の面積は約2,271平方キロメートルです。したがって、沖縄県は東京都よりも面積が大きいです。


In [10]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内に東京駅で開催されるイベント情報は以下の通りです：

1. **東急駅・富士の内 8月のお勧めイベント**
   - 開催予定のイベントをまとめた情報。詳細は[こちら](https://www.enjoytokyo.jp/event/list/area1306/its14/)で確認できます。

2. **今週末のアゲアゲに！東京のイベント**
   - 多数のイベントが紹介されています。詳細は[こちら](https://www.enjoytokyo.jp/event/list/its06/)で確認できます。

3. **東京駅の最新イベント情報**
   - 最新のイベント情報が提供されており、特に限定企画についての情報があります。詳細は[こちら](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)で確認できます。

4. **東京駅周辺のイベント情報**
   - 有名なイベントやアトラクションがリストアップされています。詳細は[こちら](https://www.walkerplus.com/event_list/ar0313/sc309880d/)を参照してください。

これらのリンクを使って、より詳細な情報を取得できます。興味のあるイベントに参加してみてください！


In [11]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:東北6県は？'

東北6県は以下の通りです：

1. 青森県 (あおもりけん)
2. 岩手県 (いわてけん)
3. 宮城県 (みやぎけん)
4. 秋田県 (あきたけん)
5. 山形県 (やまがたけん)
6. 福島県 (ふくしまけん) 

これらの県は日本の東北地方に位置しています。


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産として人気のあるもの、またはおすすめの特産品について、いくつかの情報源から以下のポイントをまとめました。

1. **宮城県の特産品**:
   - **牛タン**: 宮城県の名物料理である牛タンは、お土産としても人気があります。食感や味わいが特徴で、専用のスライサーや調味料も販売されています。
   - **萩の月**: 宮城県のスイーツとして有名な和菓子で、しっとりとした生地とクリームが絶妙なハーモニーを生み出しています。
   - **三陸の海産物**: 特に珍味である海苔や干物、貝類など、新鮮な海の幸が多く、観光客に人気です。

2. **おすすめのお土産ランキング**:
   - **じゃらん**による「宮城のオ土産ランキング」では、以下のアイテムが上位にランクインしています:
     - 1位: 牛タン
     - 2位: 萩の月
     - 3位: ずんだ餅
     - 4位: 笹かまぼこ
     - 5位: かまぼこ

3. **購入方法**:
   - 多くの特産品は、観光地や駅近くの土産物店、またはオンラインショップで購入可能です。
   - また、地元の市場や直売所でも新鮮な食品が手に入るため、訪れる際には立ち寄ってみることをおすすめします。

4. **特産品情報の収集先**:
   - 宮城県の観光案内サイトや、地元のデパート、オンラインショップで最新の情報が得られます。

詳細については、以下のリンクから確認できます。

- [じゃらんの宮城県の特産品情報](https://www.jalan.net/omiyage/040000/)
- [宮城県の特産品情報ページ](https://shunsentanbou.pref.miyagi.jp/theme/miyage/)
- [宮城県の海産物特集](https://food.biglobe.ne.jp/prefectures_tohoku_miyagi/feature_longevity/) 

このように、宮城県には多くの魅力的なお土産がありますので、ぜひ訪れた際にはお土産選びを楽しんでください。

---ご利用ありがとうございました！---
